In [9]:
!pip install torch torchvision nltk python-telegram-bot --quiet
print("Библиотеки успешно установлены!")


Библиотеки успешно установлены!


In [10]:
import json
import random
import logging
import asyncio
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.stem.snowball import SnowballStemmer
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

# Скачиваем пакет токенизации NLTK
nltk.download('punkt', quiet=True)

# Очищаем старые лог-файлы, если они были
import os
for log_file in ["training.log", "dialogs.log"]:
    if os.path.exists(log_file):
        os.remove(log_file)

# База знаний (Интенты для подбора и покупки ТВ)
intents_data = {
  "intents": [
    {
      "tag": "greeting",
      "patterns": ["Привет", "Здравствуйте", "Добрый день", "Ищу телевизор", "Помогите выбрать ТВ"],
      "responses": ["Приветствую! Я помогу вам подобрать идеальный телевизор. Какая диагональ вас интересует?", "Здравствуйте! Давайте выберем отличный ТВ. Подсказать варианты или ищете что-то конкретное?"]
    },
    {
      "tag": "goodbye",
      "patterns": ["Пока", "До свидания", "Спасибо, я пойду", "Отмена"],
      "responses": ["До свидания! Если появятся вопросы по моделям — пишите.", "Хорошего дня! Возвращайтесь за покупками."]
    },
    {
      "tag": "tv_selection",
      "patterns": ["Посоветуй телевизор", "Как выбрать ТВ?", "Какой бренд лучше?", "Какую диагональ взять в спальню?", "Нужен Смарт ТВ", "Порекомендуй 4К телевизор"],
      "responses": [
        "Для спальни отлично подойдут модели 43-55 дюймов. Для гостиной — от 65 дюймов. Вам важна поддержка Smart TV?",
        "Среди брендов сейчас лидируют Samsung, LG (OLED матрицы) и Xiaomi/TCL (лучшее соотношение цена/качество). Какой у вас бюджет?",
        "Рекомендую выбирать модели с поддержкой 4K Ultra HD и HDR для сочной картинки. Подсказать конкретную модель по вашему бюджету?"
      ]
    },
    {
      "tag": "tv_purchase",
      "patterns": ["Хочу купить", "Как оформить заказ?", "Купить телевизор", "Оформить покупку", "В корзину", "Где купить?"],
      "responses": [
        "Отличный выбор! Для оформления покупки пришлите, пожалуйста, артикул выбранного ТВ или его точное название. Наш менеджер свяжется с вами.",
        "Оформить заказ можно прямо здесь. Напишите модель телевизора, ваше имя и номер телефона для связи."
      ]
    }
  ]
}

# Записываем базу знаний в файл intents.json
with open('intents.json', 'w', encoding='utf-8') as f:
    json.dump(intents_data, f, ensure_ascii=False, indent=2)

print("Файл intents.json успешно создан в Colab!")


Файл intents.json успешно создан в Colab!


In [11]:
# 1. Текстовые утилиты
stemmer = SnowballStemmer("russian")

def tokenize(sentence):
    return nltk.word_tokenize(sentence, language="russian")

def stem(word):
    return stemmer.stem(word.lower())

def bag_of_words(tokenized_sentence, all_words):
    tokenized_sentence = [stem(w) for w in tokenized_sentence]
    bag = np.zeros(len(all_words), dtype=np.float32)
    for idx, w in enumerate(all_words):
        if w in tokenized_sentence:
            bag[idx] = 1.0
    return bag

# 2. Архитектура Feed-Forward нейросети
class NeuralNet(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(NeuralNet, self).__init__()
        self.l1 = nn.Linear(input_size, hidden_size)
        self.l2 = nn.Linear(hidden_size, hidden_size)
        self.l3 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.l1(x)
        out = self.relu(out)
        out = self.l2(out)
        out = self.relu(out)
        out = self.l3(out)
        return out

print("Компоненты обработки текста и нейросеть инициализированы!")


Компоненты обработки текста и нейросеть инициализированы!


In [12]:
import nltk
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [13]:
import logging
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk

# Автоматически скачиваем punkt_tab для токенизации (решает LookupError)
nltk.download('punkt_tab', quiet=True)

# Настройка логирования обучения
train_logger = logging.getLogger("train_logger")
train_logger.setLevel(logging.INFO)

# Чтобы логи не дублировались при повторных запусках ячейки
if train_logger.hasHandlers():
    train_logger.handlers.clear()

# Настройка вывода логов
file_handler = logging.FileHandler("training.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
train_logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
train_logger.addHandler(stream_handler)

# Загрузка и подготовка датасета
with open('intents.json', 'r', encoding='utf-8') as f:
    intents = json.load(f)

all_words = []
tags = []
xy = []

for intent in intents['intents']:
    tag = intent['tag']
    tags.append(tag)
    for pattern in intent['patterns']:
        w = tokenize(pattern)
        all_words.extend(w)
        xy.append((w, tag))

ignore_words = ['?', '!', '.', ',']
all_words = [stem(w) for w in all_words if w not in ignore_words]
all_words = sorted(list(set(all_words)))
tags = sorted(list(set(tags)))

X_train = []
y_train = []
for (pattern_sentence, tag) in xy:
    bag = bag_of_words(pattern_sentence, all_words)
    X_train.append(bag)
    label = tags.index(tag)
    y_train.append(label)

X_train = np.array(X_train)
y_train = np.array(y_train)

# Модифицированный датасет с приведением к тензорам PyTorch
class ChatDataset(Dataset):
    def __init__(self):
        self.n_samples = len(X_train)
        # Сразу переводим в тензоры float32 для совместимости с Linear слоями PyTorch
        self.x_data = torch.from_numpy(X_train).to(dtype=torch.float32)
        self.y_data = torch.from_numpy(y_train).to(dtype=torch.long)

    def __getitem__(self, index):
        return self.x_data[index], self.y_data[index]

    def __len__(self):
        return self.n_samples

# Гиперпараметры
batch_size = 8
hidden_size = 8
output_size = len(tags)      # Количество классов
input_size = len(all_words)  # Динамический размер входа (будет равен 35)
learning_rate = 0.001
num_epochs = 1000

dataset = ChatDataset()
train_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, num_workers=0)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ИСПРАВЛЕНО: Вместо num_classes передаем output_size
model = NeuralNet(input_size=input_size, hidden_size=hidden_size, num_classes=output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

train_logger.info("Начало обучения ТВ-консультанта...")

for epoch in range(num_epochs):
    for (words, labels) in train_loader:
        words = words.to(device)
        labels = labels.to(device) # Тип torch.long уже задан в ChatDataset

        outputs = model(words)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 100 == 0:
        train_logger.info(f'Эпоха [{epoch+1}/{num_epochs}], Ошибка (Loss): {loss.item():.4f}')

train_logger.info(f'Обучение завершено. Финальная ошибка: {loss.item():.4f}')

# Сохранение весов
data = {
    "model_state": model.state_dict(),
    "input_size": input_size,
    "output_size": output_size,
    "hidden_size": hidden_size,
    "all_words": all_words,
    "tags": tags
}

FILE = "data.pth"
torch.save(data, FILE)
train_logger.info(f'Файл модели успешно сохранен в {FILE}')


INFO: Начало обучения ТВ-консультанта...
INFO:train_logger:Начало обучения ТВ-консультанта...
INFO: Эпоха [100/1000], Ошибка (Loss): 0.6928
INFO:train_logger:Эпоха [100/1000], Ошибка (Loss): 0.6928
INFO: Эпоха [200/1000], Ошибка (Loss): 0.2482
INFO:train_logger:Эпоха [200/1000], Ошибка (Loss): 0.2482
INFO: Эпоха [300/1000], Ошибка (Loss): 0.0444
INFO:train_logger:Эпоха [300/1000], Ошибка (Loss): 0.0444
INFO: Эпоха [400/1000], Ошибка (Loss): 0.0130
INFO:train_logger:Эпоха [400/1000], Ошибка (Loss): 0.0130
INFO: Эпоха [500/1000], Ошибка (Loss): 0.0116
INFO:train_logger:Эпоха [500/1000], Ошибка (Loss): 0.0116
INFO: Эпоха [600/1000], Ошибка (Loss): 0.0047
INFO:train_logger:Эпоха [600/1000], Ошибка (Loss): 0.0047
INFO: Эпоха [700/1000], Ошибка (Loss): 0.0028
INFO:train_logger:Эпоха [700/1000], Ошибка (Loss): 0.0028
INFO: Эпоха [800/1000], Ошибка (Loss): 0.0011
INFO:train_logger:Эпоха [800/1000], Ошибка (Loss): 0.0011
INFO: Эпоха [900/1000], Ошибка (Loss): 0.0012
INFO:train_logger:Эпоха [900

In [14]:
import logging
import json
import random
import asyncio
import re  # Для умного поиска диагоналей, брендов, контактов и ответов
import torch
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes

# ========================================================
# 1. НАСТРОЙКА ЛОГИРОВАНИЯ ДИАЛОГОВ
# ========================================================
dialog_logger = logging.getLogger("dialog_logger")
dialog_logger.setLevel(logging.INFO)
if dialog_logger.hasHandlers():
    dialog_logger.handlers.clear()

file_handler = logging.FileHandler("dialogs.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))
dialog_logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
dialog_logger.addHandler(stream_handler)

# ========================================================
# 2. ЗАГРУЗКА ДАННЫХ И МОДЕЛИ
# ========================================================
try:
    with open('intents.json', 'r', encoding='utf-8') as f:
        intents_data = json.load(f)
except FileNotFoundError:
    print("КРИТИЧЕСКАЯ ОШИБКА: Файл 'intents.json' не найден! Проверьте вкладку файлы в Colab.")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
FILE = "data.pth"
checkpoint = torch.load(FILE, map_location=device)

model = NeuralNet(checkpoint["input_size"], checkpoint["hidden_size"], checkpoint["output_size"]).to(device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

all_words = checkpoint['all_words']
tags = checkpoint['tags']

# ========================================================
# 3. УМНАЯ ФУНКЦИЯ ПРАВИЛ С ПАМЯТЬЮ И ЗАКАЗОМ (Исправлено)
# ========================================================
def check_hard_rules(text, user_data):
    text_lower = text.strip().lower()

    # ЭТАП 4: Если бот уже ждет контактные данные от пользователя
    if user_data.get('status') == 'waiting_for_contacts':
        phone_match = re.search(r'(\+?\d[0-9\s\-\(\)]{6,14}\d)', text_lower)
        if phone_match or len(text_lower) > 5:
            user_data.clear() # Очищаем сессию ПОСЛЕ получения контактов
            return ("✅ Спасибо! Ваша заявка принята. Менеджер свяжется с вами по указанным контактам в "
                    "течение 15 минут для подтверждения заказа и уточнения адреса доставки. Хорошего дня!")

    # ЭТАП 3: Если выбор был сделан на прошлом шаге, и пользователь соглашается на оформление
    if user_data.get('status') == 'waiting_for_confirmation':
        if any(word in text_lower for word in ['оформ', 'заказ', 'да', 'давай', 'хочу', 'купить', 'подтвержд']):
            user_data['status'] = 'waiting_for_contacts' # Переключаем статус на сбор данных
            return "Отлично! Пожалуйста, напишите ваше Имя и Номер телефона, чтобы менеджер мог связаться с вами для оформления заказа."
        elif 'отзыв' in text_lower:
            return f"Модель {user_data.get('brand')} на {user_data.get('size')}\" имеет рейтинг 4.8/5. Покупатели хвалят её за шустрый смарт и сочную картинку. Оформляем покупку?"

    # НОВОЕ ПРАВИЛО: Перехват абстрактных описаний "большой", "умный" в самом начале диалога
    if not user_data.get('size') and not user_data.get('brand'):
        if any(word in text_lower for word in ['большой', 'крупный', 'огромный', 'умный', 'смарт']):
            return ("Отличный запрос! Под понятие 'большой и умный' идеально подходят современные 4K-телевизоры "
                    "со Smart TV. Подскажите, какую диагональ экрана вы рассматриваете? Обычно для гостиной "
                    "выбирают модели на 50, 55 или 65 дюймов.")

    # ЭТАП 1: Поиск параметров (Диагональ и Бренд)
    size_match = re.search(r'\b([2-9]\d)\b', text_lower)
    if size_match:
        user_data['size'] = size_match.group(1)

    if 'lg' in text_lower:
        user_data['brand'] = 'LG'
    elif 'samsung' in text_lower:
        user_data['brand'] = 'Samsung'
    elif 'xiaomi' in text_lower:
        user_data['brand'] = 'Xiaomi'
    elif 'tcl' in text_lower:
        user_data['brand'] = 'TCL'

    # ЭТАП 2: Если собрали и бренд, и диагональ — предлагаем сделку
    if user_data.get('brand') and user_data.get('size') and not user_data.get('status'):
        user_data['status'] = 'waiting_for_confirmation' # Замораживаем и ждем согласия
        brand = user_data['brand']
        size = user_data['size']
        return (f"🎉 Отлично, выбор сделан! Мы подобрали вам телевизор {brand} с диагональю {size} дюймов.\n\n"
                f"Это идеальный вариант под ваши критерии с официальной гарантией. "
                f"Оформляем заказ или хотите сначала посмотреть отзывы?")

    # Базовые правила (перехват комнат, общих вопросов)
    if 'спальн' in text_lower or 'кухн' in text_lower:
        return "Для спальни или кухни идеальным выбором будут компактные и практичные диагонали 32\" или 43\" дюйма. Какой размер вам кажется более подходящим?"

    if 'гостин' in text_lower or 'зал' in text_lower:
        return "В гостиную обычно берут экраны покрупнее, чтобы было комфортно смотреть всей семьей — от 50\", 55\" или даже 65\" дюймов. К какому размеру склоняетесь?"

    if text_lower in ['да', 'важна', 'конечно', 'да, важна', 'нужна', 'да со смарт тв', 'да со смарт', 'давай']:
        return "Отлично! Давайте сузим поиск. Подскажите, какую диагональ экрана вы рассматриваете (например: 43, 55 дюймов) или какой у вас бренд в приоритете?"

    if size_match and not user_data.get('brand'):
        return f"Понял, сохранил: смотрим ТВ с диагональю {user_data['size']} дюймов. Какой бренд вам ближе (Samsung, LG, Xiaomi, TCL), чтобы я подобрал лучшие модели?"

    if user_data.get('brand') and not user_data.get('size'):
        brand = user_data['brand']
        return f"Телевизоры {brand} — прекрасный выбор! Какую диагональ экрана для него мы ищем? (Например: 43, 55, 65 дюймов)"

    if 'какие' in text_lower and ('диагонал' in text_lower or 'размер' in text_lower):
        return ("Обычно выбирают из таких стандартов:\n"
                "• Для кухни/спальни: 32\" или 43\" дюйма.\n"
                "• Универсальные для гостиной: 50\" и 55\" дюймов.\n"
                "• Для домашнего кинотеатра: 65\", 75\" и выше.\n"
                "Какая комната у вас, чтобы подобрать точнее?")

    if re.search(r'\b(\d{3,6})\b', text_lower):
        return "Отличный бюджет! В этой ценовой категории можно подобрать как надежный базовый вариант от Xiaomi/TCL, так и более технологичный LG или Samsung. Какой бренд или диагональ экрана вы хотите рассмотреть?"

    return None

# ========================================================
# 4. ЛОГИКА ОТВЕТОВ БОТА
# ========================================================
async def handle_message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_text = update.message.text
    user_id = update.message.from_user.id
    user_name = update.message.from_user.username or "NoUsername"

    # Шаг 1: Проверяем текст через жесткие правила с памятью
    reply_text = check_hard_rules(user_text, context.user_data)
    tag = "Hard_Rule_Triggered"
    prob_val = 1.00

    # Шаг 2: Если жесткие правила молчат — отправляем текст в нейросеть
    if not reply_text:
        sentence = tokenize(user_text)
        X = bag_of_words(sentence, all_words)
        X = X.reshape(1, -1)
        X = torch.from_numpy(X).to(dtype=torch.float32).to(device)

        outputs = model(X)
        _, predicted = torch.max(outputs, dim=1)
        tag = tags[predicted.item()]

        probs = torch.softmax(outputs, dim=1).squeeze()
        prob = probs[predicted.item()]
        prob_val = prob.item()

        # ИСПРАВЛЕНО: Подняли порог до 0.70, чтобы отсечь ложные срабатывания вроде Conf: 0.66
        if prob_val > 0.70:
            for intent in intents_data['intents']:
                if tag == intent["tag"]:
                    reply_text = random.choice(intent['responses'])
                    break

    if not reply_text:
        reply_text = "Я пока только учусь подбирать телевизоры. Подскажите, пожалуйста: вы ищете конкретный бренд (Samsung, LG, Xiaomi, TCL) или ориентируетесь по размеру экрана?"

    # Записываем лог диалога
    dialog_logger.info(f"User (ID: {user_id}, @{user_name}): '{user_text}' -> Bot: '{reply_text}' [Tag: {tag}, Conf: {prob_val:.2f}]")
    await update.message.reply_text(reply_text)

async def start_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    context.user_data.clear() # Полностью сбрасываем память при /start
    await update.message.reply_text("Привет! Я ваш персональный помощник по подбору телевизоров. Напишите мне, какой ТВ вы ищете?")

# ========================================================
# 5. АСИНХРОННЫЙ ЗАПУСК БОТА
# ========================================================
async def run_bot_in_colab():
    TOKEN = 'ВАШ_ТЕЛЕГРАМ_ТОКЕН'

    if TOKEN == "ВАШ_ТЕЛЕГРАМ_ТОКЕН":
        print("ОШИБКА: Замените токен!")
        return

    app = Application.builder().token(TOKEN).build()
    app.add_handler(CommandHandler("start", start_command))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle_message))

    await app.initialize()
    await app.updater.start_polling()
    await app.start()

    print("Бот перезапущен с исправлением ложных срабатываний!")
    print("Чтобы остановить бота, нажмите кнопку СТОП (квадрат) на этой ячейке.")

    try:
        while True:
            await asyncio.sleep(1)
    except asyncio.CancelledError:
        print("\nПолучен сигнал остановки ячейки...")
    finally:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()
        print("Бот полностью остановлен.")

await run_bot_in_colab()


Бот перезапущен с исправлением ложных срабатываний!
Чтобы остановить бота, нажмите кнопку СТОП (квадрат) на этой ячейке.

Получен сигнал остановки ячейки...
Бот полностью остановлен.


In [15]:

from google.colab import files

# Скачивание логов обучения и диалогов
try:
    files.download('training.log')
    files.download('dialogs.log')
    print("Файлы логов успешно отправлены на скачивание.")
except FileNotFoundError as e:
    print(f"Ошибка: файл не найден на диске. {e}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Файлы логов успешно отправлены на скачивание.
